# Simple Agent (Notebook Version)

Notebook edition of `agent.py` — a minimal Claude-powered agent with a manual tool-use loop, running on `claude-haiku-4-5`.

Run the cells in order. The last cell gives you a chat loop; re-running the "send a message" cell keeps the conversation going since `agent.messages` persists in the notebook's memory.

## 1. Install dependencies

In [ ]:
%pip install -q anthropic

## 2. Set your API key

Uses `getpass` so the key is typed into a hidden prompt instead of being saved in plain text in the notebook file.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your ANTHROPIC_API_KEY: ")

## 3. Define tools, the agent loop, and the `Agent` class

In [ ]:
import datetime

import anthropic

MODEL = "claude-haiku-4-5"
MAX_TOKENS = 16000

SYSTEM_PROMPT = (
    "You are a helpful assistant with access to tools. "
    "Use them when they help answer the user's request; otherwise reply directly."
)

TOOLS = [
    {
        "name": "get_current_time",
        "description": "Get the current date and time.",
        "input_schema": {
            "type": "object",
            "properties": {},
        },
    },
    {
        "name": "calculator",
        "description": "Evaluate a basic arithmetic expression, e.g. '2 + 2 * 3'.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "An arithmetic expression using +, -, *, /, and parentheses.",
                },
            },
            "required": ["expression"],
        },
    },
]


def get_current_time() -> str:
    return datetime.datetime.now().isoformat()


def calculator(expression: str) -> str:
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        return "Error: expression contains disallowed characters."
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as exc:
        return f"Error: {exc}"


def execute_tool(name: str, tool_input: dict) -> str:
    if name == "get_current_time":
        return get_current_time()
    if name == "calculator":
        return calculator(tool_input["expression"])
    return f"Error: unknown tool '{name}'"


class Agent:
    """A minimal conversational agent that can call tools in a loop."""

    def __init__(self, client: anthropic.Anthropic | None = None):
        self.client = client or anthropic.Anthropic()
        self.messages: list[dict] = []

    def send(self, user_input: str) -> str:
        self.messages.append({"role": "user", "content": user_input})

        while True:
            response = self.client.messages.create(
                model=MODEL,
                max_tokens=MAX_TOKENS,
                system=SYSTEM_PROMPT,
                tools=TOOLS,
                messages=self.messages,
            )
            self.messages.append({"role": "assistant", "content": response.content})

            if response.stop_reason != "tool_use":
                break

            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = execute_tool(block.name, block.input)
                    tool_results.append(
                        {
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": result,
                        }
                    )
            self.messages.append({"role": "user", "content": tool_results})

        return next(
            (block.text for block in response.content if block.type == "text"), ""
        )

## 4. Create the agent

In [ ]:
agent = Agent()

## 5. Send a message

Edit the string below and re-run this cell to keep chatting — `agent.messages` keeps the history between runs, so context carries over.

In [ ]:
reply = agent.send("Hi! What's 12 * 7, and what time is it right now?")
print(reply)

## 6. Optional: interactive chat loop

Run this cell to chat back and forth in the notebook's input prompt. Type `exit` to stop.

In [ ]:
while True:
    user_input = input("You: ")
    if user_input.strip().lower() in {"exit", "quit"}:
        break
    print(f"Agent: {agent.send(user_input)}")